# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shakhaoathpappu-jpg/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# SETUP — discover what's actually in the warehouse before assuming any column names
!pip install -q huggingface_hub pandas pyarrow scikit-learn

from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "FlyRank/internship-warehouse"

api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)
for f in files:
    print(f)

SecretNotFoundError: Secret HF_TOKEN does not exist.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Config used throughout this notebook — write your text answer in the markdown cell above, not here.
LANE = "content_refresh_prioritization"
MID_PANEL_MONTH = "2026-03"
PRIOR_MONTH = "2026-02"       # needed for the trend-direction feature
SEALED_TEST_MONTH = "2026-06" # never touch this for label logic


## Unit of analysis + time window

One row represents one webpage for one monthly observation.

I will use the Content Refresh Prioritization lane.

The analysis will use a mid-panel month (2026-03) to avoid using the final month as the outcome window.

The goal is to rank webpages that are most likely to need refreshing based on their search performance.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: feature / label / context / excluded

**Features**
- Clicks
- Impressions
- CTR
- Average Position
- Trend Direction

**Label / Proxy**
- Refresh Priority Score (or ranking priority)

**Context**
- Month
- Page

**Excluded**
- Future outcome information and any label-derived columns.

Reason: They would introduce data leakage and make the evaluation unrealistic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the table for this lane.
# ⚠️ Replace DATA_FILE with the actual filename you saw printed in the setup cell above.
DATA_FILE = "PUT_THE_CORRECT_FILENAME_HERE.parquet"

local_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=DATA_FILE, token=HF_TOKEN)
df = pd.read_parquet(local_path)  # swap to pd.read_csv(local_path) if the file is .csv

print(df.shape)
df.head()


In [ ]:
# QUERY 1 — GRAIN: one row really is one (page, month)?
month_df = df[df["month"] == MID_PANEL_MONTH].copy()
dupe_count = month_df.duplicated(subset=["page"]).sum()

print(f"Rows in {MID_PANEL_MONTH}: {len(month_df)}")
print(f"Duplicate 'page' rows: {dupe_count}")
assert dupe_count == 0, "Grain claim failed — page is not unique per month in this table."

In [ ]:
# QUERY 2 — ROW COUNT + DATE SPAN for this lane's slice
print("Total rows (all months):", len(df))
print("Date span:", df["month"].min(), "to", df["month"].max())
print(f"Rows in mid-panel month {MID_PANEL_MONTH}:", len(month_df))

In [ ]:
# QUERY 3 — AVAILABILITY: filter with IS TRUE, show how many rows survive
# ⚠️ Replace 'is_available' with the actual boolean/flag column name in your table.
available_df = month_df[month_df["is_available"] == True].copy()

dropped = len(month_df) - len(available_df)
print(f"Rows before availability filter: {len(month_df)}")
print(f"Rows after is_available IS TRUE:  {len(available_df)}")
print(f"Dropped: {dropped} rows ({dropped / len(month_df):.1%})")

### Five features (max) — each with an "available when?" line

1. **Clicks** — knowable at the decision moment because it's the closed count of clicks recorded through the end of the decision month.
2. **Impressions** — knowable at the decision moment for the same reason: it's a closed monthly total, not a forward-looking number.
3. **CTR** — knowable because it's simply clicks ÷ impressions, both already closed for the month.
4. **Average Position** — knowable because Search Console reports it as a completed monthly average, not a live/future estimate.
5. **Trend Direction** — knowable because it only compares the decision month to the *prior* month, never to a future one.

In [ ]:
# Build the feature frame.
# ⚠️ Adjust column names (clicks / impressions / ctr / avg_position) to match your table.
features_df = available_df[["page", "month", "clicks", "impressions", "ctr", "avg_position"]].copy()

prior_df = df[df["month"] == PRIOR_MONTH][["page", "clicks"]].rename(columns={"clicks": "prior_clicks"})
features_df = features_df.merge(prior_df, on="page", how="left")

features_df["trend_direction"] = (features_df["clicks"] - features_df["prior_clicks"]).apply(
    lambda x: "up" if x > 0 else ("down" if x < 0 else "flat")
)

features_df.head()

### The trap — add ONE label-derived column on purpose

We define a simple proxy label ("needs refresh" = clicks fell month-over-month), train an honest model, then sneak in a column that is basically the label restated as a feature, and watch the score jump toward perfect. Then we delete it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

features_df["needs_refresh"] = (features_df["trend_direction"] == "down").astype(int)
y = features_df["needs_refresh"]

# --- Honest version: only legitimately-available features ---
X_honest = features_df[["clicks", "impressions", "ctr", "avg_position"]].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print(f"Honest AUC (no leakage): {honest_auc:.3f}")

# --- Leaky version: sneak in a column derived from the label itself ---
X_leaky = X_honest.copy()
X_leaky["clicks_delta"] = features_df["clicks"] - features_df["prior_clicks"].fillna(0)  # this IS the label, restated

Xl_train, Xl_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
leaky_model = LogisticRegression(max_iter=1000).fit(Xl_train, y_train)
leaky_auc = roc_auc_score(y_test, leaky_model.predict_proba(Xl_test)[:, 1])
print(f"Leaky AUC (with 'clicks_delta'): {leaky_auc:.3f}")

print(f"\nScore jumped from {honest_auc:.3f} to {leaky_auc:.3f} just by adding a column that encodes the label.")

# Delete the leaky column — keep the honest number as the real one.
del X_leaky
print(f"\nLeaky column removed. The honest AUC ({honest_auc:.3f}) is the number we keep.")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

- This dataset cannot prove causal relationships — it shows association between search signals and refresh need, not proof that refreshing *causes* better performance.
- The analysis provides decision support based on observed search performance, not a guarantee.
- Historical data may be incomplete for some pages (e.g. newly published pages have fewer prior months to compute a trend from), and future search behavior can't be guaranteed from past observations.
- **Named limitation of this slice:** pages with fewer than one prior month of history can't get a `trend_direction` feature at all, so this lane's ranking is systematically less reliable for newly published or newly indexed pages.
- Results should be read as directional/decision-support, not as absolute predictions.

In [ ]:
# Quick, honest look at missing values — backs up the named limitation above.
print("Missing values in the feature frame:")
print(features_df.isnull().sum())

no_prior_month = features_df["prior_clicks"].isnull().sum()
print(f"\nPages with no prior-month data (can't compute trend_direction): {no_prior_month}")

## Data limits

This dataset cannot prove causal relationships.

The analysis provides decision support based on observed search performance.

Historical data may be incomplete, and future search behavior cannot be guaranteed from past observations.

The results should be interpreted as directional rather than absolute predictions.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.